# Type2 HFSS Import

이 노트북은 기존 runtime을 직접 호출하는 thin manual notebook이다. `examples/type2_fixed.toml`에서 scene STEP ledger를 export하고, 기존 AEDT desktop에 attach해서 import runtime을 실행한 뒤 결과 `.aedt`와 imported ledger를 확인한다.

- STEP export runtime: `entry.generate_type2_step.export_type2_step_artifacts(...)`
- HFSS import runtime: `peetsfea.backend.pyaedt.type2_step_import_pipeline.import_type2_step_ledger_into_hfss(...)`
- canonical scene STEP: `run/step/type2/type2_scene.step`
- 결과물: `run/aedt/type2_step_import/type2_import.aedt`, `run/aedt/type2_step_import/type2_imported_ledger.json`
- import 후 material/color/transparency 적용 책임은 runtime에 있다.


In [1]:
from __future__ import annotations

import json
from pathlib import Path
from pprint import pprint
import subprocess
import sys

repo_root_result = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"],
    check=True,
    capture_output=True,
    text=True,
)
repo_root_text = repo_root_result.stdout.strip()
if repo_root_text == "":
    raise RuntimeError("git rev-parse --show-toplevel returned empty stdout")
REPO_ROOT = Path(repo_root_text).resolve()
pyproject_path = REPO_ROOT / "pyproject.toml"
if not pyproject_path.is_file():
    raise FileNotFoundError(f"repo root is missing pyproject.toml: {pyproject_path}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from entry.generate_type2_step import export_type2_step_artifacts
from peetsfea.aedt import Hfss
from peetsfea.backend.pyaedt.type2_step_import_pipeline import import_type2_step_ledger_into_hfss

TYPE2_TOML_PATH = REPO_ROOT / "examples" / "type2_fixed.toml"
STEP_OUTPUT_DIR = REPO_ROOT / "run" / "step" / "type2"
STEP_LEDGER_PATH = STEP_OUTPUT_DIR / "type2_step_ledger.json"
TYPE2_SCENE_STEP_PATH = STEP_OUTPUT_DIR / "type2_scene.step"
OUTPUT_AEDT_PATH = REPO_ROOT / "run" / "aedt" / "type2_step_import" / "type2_import.aedt"
IMPORTED_LEDGER_PATH = REPO_ROOT / "run" / "aedt" / "type2_step_import" / "type2_imported_ledger.json"
DESIGN_NAME = "type2_step_import"

print(f"repo root: {REPO_ROOT}")
print(f"type2 TOML: {TYPE2_TOML_PATH}")
print(f"scene STEP: {TYPE2_SCENE_STEP_PATH}")
print(f"STEP ledger: {STEP_LEDGER_PATH}")
print(f"AEDT output: {OUTPUT_AEDT_PATH}")
print(f"imported ledger: {IMPORTED_LEDGER_PATH}")
print("AEDT attach mode: GUI-visible existing desktop")


repo root: /home/harry/Projects/PythonProjects/peetsfea-main
type2 TOML: /home/harry/Projects/PythonProjects/peetsfea-main/examples/type2_fixed.toml
scene STEP: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2/type2_scene.step
STEP ledger: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2/type2_step_ledger.json
AEDT output: /home/harry/Projects/PythonProjects/peetsfea-main/run/aedt/type2_step_import/type2_import.aedt
imported ledger: /home/harry/Projects/PythonProjects/peetsfea-main/run/aedt/type2_step_import/type2_imported_ledger.json
AEDT attach mode: GUI-visible existing desktop


## 1. Export Type2 scene STEP ledger

이 셀은 `export_type2_step_artifacts(...)`를 그대로 호출해서 canonical single scene STEP와 `type2_step_ledger.json`을 만든다. notebook 안에 별도 export 로직은 두지 않는다.


In [2]:
step_ledger = export_type2_step_artifacts(
    toml_path=TYPE2_TOML_PATH,
    output_dir=STEP_OUTPUT_DIR,
    ledger_path=STEP_LEDGER_PATH,
    seed=0,
)

print(f"source TOML: {step_ledger['source_toml_path']}")
print(f"output dir: {step_ledger['output_dir']}")
print(f"scene STEP: {step_ledger['scene_step_path']}")
print(f"seed: {step_ledger['seed']}")
print(f"non-model object count: {len(step_ledger['non_model_objects'])}")
print(f"modeled object count: {len(step_ledger['modeled_objects'])}")


source TOML: /home/harry/Projects/PythonProjects/peetsfea-main/examples/type2_fixed.toml
output dir: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2
scene STEP: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2/type2_scene.step
seed: 0
non-model object count: 1
modeled object count: 2


## 2. Import Into Existing HFSS Desktop

이 셀은 `Hfss(..., non_graphical=False, new_desktop=False)`로 기존 AEDT desktop에 attach한 뒤 `import_type2_step_ledger_into_hfss(...)`를 직접 호출한다. import 후 material/color/transparency 적용과 detach release는 runtime이 수행한다.


In [3]:
imported_ledger = import_type2_step_ledger_into_hfss(
    hfss=Hfss(project=None, design=DESIGN_NAME, non_graphical=False, new_desktop=False),
    step_ledger_path=STEP_LEDGER_PATH,
    output_aedt_path=OUTPUT_AEDT_PATH,
    imported_ledger_path=IMPORTED_LEDGER_PATH,
)

print(f"AEDT path: {imported_ledger['aedt_path']}")
print(f"imported ledger: {imported_ledger['imported_ledger_path']}")
print(f"non-model imported objects: {len(imported_ledger['non_model_objects'])}")
print(f"modeled imported objects: {len(imported_ledger['modeled_objects'])}")
print("Runtime detached the notebook HFSS handle after import/save.")


PyAEDT INFO: Python version 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:16:04) [GCC 11.2.0].
PyAEDT INFO: PyAEDT version 0.25.1.
PyAEDT INFO: Initializing Desktop session.
PyAEDT INFO: AEDT version 2025.2.
PyAEDT INFO: New AEDT session is starting on gRPC port 51535.
PyAEDT INFO: Starting new AEDT gRPC session on port 51535.
PyAEDT INFO: Launching AEDT server with gRPC transport mode: TransportMode.UDS
PyAEDT INFO: Electronics Desktop started on gRPC port 51535 after 9.7 seconds.
PyAEDT INFO: AEDT installation Path /opt/ansys_inc/v252/AnsysEM
PyAEDT INFO: Connected to AEDT gRPC session on port 51535.
PyAEDT WARNING: Service Pack is not detected. PyAEDT is currently connecting in Insecure Mode.
PyAEDT WARNING: Please download and install latest Service Pack to use connect to AEDT in Secure Mode.
PyAEDT INFO: Project Project64 has been created.
PyAEDT INFO: Added design 'type2_step_import' of type HFSS.
PyAEDT INFO: AEDT objects correctly read
PyAEDT INFO: Modeler class

ValueError: scene STEP import is missing required modeled body name (object_id=tx_rect_void_coil, body_name=tx_port_sheet)

## 3. Inspect Imported Ledger

이 셀은 runtime이 쓴 imported ledger를 읽어서 import 결과만 요약한다. notebook은 import semantics나 styling logic를 재구현하지 않는다.


In [ ]:
payload = json.loads(IMPORTED_LEDGER_PATH.read_text(encoding="utf-8"))

summary = {
    "aedt_path": payload["aedt_path"],
    "source_step_ledger_path": payload["source_step_ledger_path"],
    "scene_step_path": payload["scene_step_path"],
    "non_model_count": len(payload["non_model_objects"]),
    "modeled_count": len(payload["modeled_objects"]),
}
pprint(summary)

for group_name in ("non_model_objects", "modeled_objects"):
    print()
    print(group_name)
    print("-" * len(group_name))
    for entry in payload[group_name]:
        print(f"object_id: {entry['object_id']}")
        print(f"  role: {entry['role']}")
        print(f"  model_state: {entry['model_state']}")
        print(f"  imported_object_names: {entry['imported_object_names']}")
